In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import logging
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf

from pepo.utils import constants, set_seed

OmegaConf.register_new_resolver(
    "pepo.constants",
    lambda name: getattr(constants, name),
)


In [ ]:
config_path = Path("configs").absolute()
config_name = "chat.yaml"

with initialize_config_dir(config_dir=str(config_path), version_base="1.1"):
    cfg = compose(config_name=config_name)


original_work_dir = Path.cwd()

log_level_str = cfg.get("log_level", "INFO").upper()
log_level = getattr(logging, log_level_str, logging.INFO)

logger = instantiate(
    cfg.logger,
    log_dir=str(original_work_dir / "logs"),
    level=log_level,
)

resolved_cfg = OmegaConf.to_container(cfg, resolve=True)
logger.info("PEPO Training - Starting")
logger.info(f"Configuration:\n{OmegaConf.to_yaml(resolved_cfg)}")

set_seed(cfg.seed)
logger.info(f"Random seed set to: {cfg.seed}")

device_manager = instantiate(
    cfg.device,
    logger=logger,
)

hub_manager = instantiate(
    cfg.hub,
    logger=logger,
    load_epochs=1,
)

model = instantiate(
    cfg.model,
    logger=logger,
    device_manager=device_manager,
    hub_manager=hub_manager,
)

In [ ]:
from pepo.utils import DataManager
import os

# cache_dir = $SCRATCH/pepo/cache 
cache_dir = os.getenv("SCRATCH") + "/pepo/cache"
print(f"Cache dir: {cache_dir}")

dataset_id = "HuggingFaceH4/ultrafeedback_binarized"
split = "train_sft"

tokenizer = model.get_tokenizer()

data_manager = DataManager(
    dataset_id=dataset_id,
    split=split,
    train_split=0.99,
    eval_split=0.01,
    seed=cfg.seed,
    n_splits=cfg.model.num_networks,
    max_length=1024,
    max_prompt_length=512,
    tokenizer=tokenizer,
    cache_dir=cache_dir,
)
dataloader = data_manager.get_dataloader(
    model_idx=0,
    partition="train",
    batch_size=2,
)
batch = next(iter(dataloader))

chosen_encoded = batch["chosen_input_ids"]
chosen_att_mask = batch["chosen_attention_mask"]
chosen_resp_mask = batch["chosen_response_mask"]
reject_encoded = batch["rejected_input_ids"]
reject_att_mask = batch["rejected_attention_mask"]
reject_resp_mask = batch["rejected_response_mask"]

for b in range(batch["prompt_input_ids"].shape[0]):
    _, T_c = chosen_att_mask.shape # B, T_c
    _, T_r = reject_att_mask.shape # B, T_r
    chosen_ids = chosen_encoded[b]
    chosen_tokens = tokenizer.convert_ids_to_tokens(chosen_ids)
    for i in range(T_c):
        print(chosen_att_mask[b][i].item(), end=' ')
        print(chosen_resp_mask[b][i].item(), end=' ')
        print(f"{chosen_ids[i].item():>5}", end=' ')
        print(chosen_tokens[i], end=' ')
        print()
    print("-" * 80)

    reject_ids = reject_encoded[b]
    reject_tokens = tokenizer.convert_ids_to_tokens(reject_ids)
    for i in range(T_r):
        print(reject_att_mask[b][i].item(), end=' ')
        print(reject_resp_mask[b][i].item(), end=' ')
        print(f"{reject_ids[i].item():>5}", end=' ')
        print(reject_tokens[i], end=' ')
        print()
    print("=" * 80)


# TODO(adam): llama puts two <eot> tokens at the end of the sequence 

In [ ]:
prompts = ["Hello, how are you? I am Adam. What is your name?"]
apply_chat_template = True
tokenizer = model.get_tokenizer()

if apply_chat_template:
    formated_prompts = []
    for prompt in prompts:
        formatted_prompt = [
            {"role": "user", "content": prompt},
        ]
        formated_prompt = tokenizer.apply_chat_template(
            formatted_prompt, tokenize=False, add_generation_prompt=True
        )
        formated_prompts.append(formated_prompt)
else:
    formated_prompts = prompts

prev_padding_side = tokenizer.padding_side
tokenizer.padding_side = "left"

inputs = tokenizer(
    formated_prompts,
    return_tensors="pt",
    padding=True,
)
tokenizer.padding_side = prev_padding_side
input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]
input_ids, attention_mask = model.generate(input_ids, attention_mask, max_length=100, greedy_sampling=True, top_p_sampling=False)

In [ ]:

print(f"Input ids: {input_ids}")
print(f"Attention mask: {attention_mask}")
tokenizer = model.get_tokenizer()
output = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(f"Generated sequence idx=0:\n{output}")

In [ ]:
batch = next(iter(dataloader))
batch
# generate from batch prompt

input_ids, attention_mask = model.generate(batch["prompt_input_ids"], max_length=500)

In [ ]:
for i in range(input_ids.shape[0]):
    # print(f"Input ids: {input_ids[i]}")
    # print(f"Attention mask: {attention_mask[i]}")
    tokenizer = model.get_tokenizer()
    # for j in range(len(input_ids[i])):
    #     print(f"{input_ids[i][j].item():>5}", end=' ')
    #     print(f"{output[j]}", end=' ')
    #     print()
    # include only attention mask tokens
    input_ids_masked = [input_ids[i][j] for j in range(len(input_ids[i])) if attention_mask[i][j].item() == 1]
    # cut the sequence to the first <eot> token
    if tokenizer.eos_token_id in input_ids_masked:
        idx_eot = input_ids_masked.index(tokenizer.eos_token_id)
    else:
        idx_eot = len(input_ids[i])
    output = tokenizer.decode(input_ids_masked[:idx_eot], skip_special_tokens=True)

        
    print(f"Generated sequence idx={i}:\n{output}") 
    print("-" * 80)